# FunctorFlow KET Block Duality Demo

This notebook ports the smallest reusable core of `scripts/ket_experiments`
into `FunctorFlow/` so the left-Kan/right-Kan story becomes concrete on one
shared language-modeling task.

- `left_kan`: future block prediction from causal context
- `right_kan`: denoising / completion of a corrupted future block
- a real Transformer / GT / KET language-model comparison on PTB, Wiki-2, or Wiki-103


In [ ]:
from pathlib import Path
import sys

def _bootstrap_functorflow() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        package_root = candidate / "FunctorFlow"
        if (package_root / "__init__.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError(
        "Could not find the FunctorFlow package. Run this notebook from the repo "
        "root or from FunctorFlow/notebooks."
    )

REPO_ROOT = _bootstrap_functorflow()
print(f"FunctorFlow repo root: {REPO_ROOT}")


In [ ]:
import torch

from FunctorFlow.ket_block_duality import (
    KETBlockDualityConfig,
    build_left_kan_block_diagram,
    build_right_kan_denoise_diagram,
    run_ket_block_duality_demo,
)

left_diagram = build_left_kan_block_diagram()
right_diagram = build_right_kan_denoise_diagram()
print(left_diagram.summary())
print()
print(right_diagram.summary())


## Why This Matters

Both tasks use the same KET-style backbone, but the output semantics differ:

- left Kan predicts forward block offsets from contextualized hidden states
- right Kan completes a noisy future block under a denoising condition

The second half of the notebook then switches from this architectural duality
demo to a real corpus-level model comparison, so the same tutorial artifact can
connect the categorical picture to executable LM behavior.


In [ ]:
from FunctorFlow.ket_lm import pick_device

config = KETBlockDualityConfig(
    corpus_name="ptb",
    seq_len=32,
    batch_size=4,
    steps=2,
    block_size=4,
    num_denoise_steps=6,
    seed=0,
)
device = pick_device("cuda")
result = run_ket_block_duality_demo(config, device=device)
result["corpus"], result["config"]


In [ ]:
{
    "left_kan_block_ppl": round(result["left_kan"]["eval"]["block_ppl"], 2),
    "left_kan_offset@1": round(result["left_kan"]["eval"]["offset_accuracy"][1], 3),
    "right_kan_block_ppl": round(result["right_kan"]["eval"]["block_ppl"], 2),
    "right_kan_offset@1": round(result["right_kan"]["eval"]["offset_accuracy"][1], 3),
}


## Real Language-Model Comparison

This section now mirrors the structured language-model comparison reported in
the book, rather than the generic autoregressive LM comparison.

Set `corpus_name` to `"ptb"`, `"wiki-2"`, or `"wiki-103"`.

- use `model_profile="smoke"` for a quick validation run
- use `model_profile="reference"` for the book-style `B=4`, context `=128`
  structured-LM setup
- on a CUDA machine, `pick_device("cuda")` will automatically target the GPU


In [ ]:
import matplotlib.pyplot as plt

from FunctorFlow.structured_lm import (
    StructuredLMComparisonConfig,
    compare_structured_language_models,
)

corpus_name = "ptb"
model_profile = "smoke"  # switch to "reference" for longer benchmark-style runs
comparison_config = (
    StructuredLMComparisonConfig.historical_smoke(corpus_name)
    if model_profile == "smoke"
    else StructuredLMComparisonConfig.historical_reference(corpus_name)
)
comparison = compare_structured_language_models(comparison_config, device=device)
comparison["corpus"], comparison["config"]


In [ ]:
{
    model_name: {
        "first_ppl": round(model_result["eval"]["first_offset_ppl"], 2),
        "block_ppl": round(model_result["eval"]["block_ppl"], 2),
    }
    for model_name, model_result in comparison["models"].items()
}


## Book Reference Table

The next cell hard-codes the structured language-model results reported in the
book chapter, so the current run can be compared directly against the published
numbers.


In [ ]:
book_results = {
    "ptb": {
        "TF-Block-4": {"first_ppl": 61.13, "block_ppl": 201.49},
        "KET-Block-4": {"first_ppl": 59.52, "block_ppl": 198.92},
        "TF-Denoise-4": {"first_ppl": 3.14, "block_ppl": 3.95},
        "KET-Denoise-4": {"first_ppl": 3.12, "block_ppl": 3.94},
    },
    "wiki-2": {
        "TF-Block-4": {"first_ppl": 112.44, "block_ppl": 266.96},
        "KET-Block-4": {"first_ppl": 109.86, "block_ppl": 262.08},
        "TF-Denoise-4": {"first_ppl": 3.44, "block_ppl": 4.11},
        "KET-Denoise-4": {"first_ppl": 3.39, "block_ppl": 4.08},
    },
    "wiki-103": {
        "TF-Block-4": {"first_ppl": 455.83, "block_ppl": 782.74},
        "KET-Block-4": {"first_ppl": 416.00, "block_ppl": 748.04},
        "TF-Denoise-4": {"first_ppl": 5.44, "block_ppl": 5.75},
        "KET-Denoise-4": {"first_ppl": 5.37, "block_ppl": 5.70},
    },
}

merged_rows = []

reference_for_corpus = book_results[comparison["corpus"]]
for model_name in comparison["models"]:
    measured = comparison["models"][model_name]["eval"]
    reference = reference_for_corpus[model_name]
    delta_first = measured["first_offset_ppl"] - reference["first_ppl"]
    delta_block = measured["block_ppl"] - reference["block_ppl"]
    merged_rows.append(
        {
            "model": model_name,
            "book_first_ppl": round(reference["first_ppl"], 2),
            "book_block_ppl": round(reference["block_ppl"], 2),
            "run_first_ppl": round(measured["first_offset_ppl"], 2),
            "run_block_ppl": round(measured["block_ppl"], 2),
            "delta_first_ppl": round(delta_first, 2),
            "delta_block_ppl": round(delta_block, 2),
            "pct_error_first": round(100.0 * delta_first / reference["first_ppl"], 2),
            "pct_error_block": round(100.0 * delta_block / reference["block_ppl"], 2),
        }
    )

try:
    import pandas as pd

    comparison_df = pd.DataFrame(merged_rows)

    def highlight_delta(value):
        if value < 0:
            return "background-color: #d8f5d0"
        if value > 0:
            return "background-color: #f8d7da"
        return ""

    styler = comparison_df.style.format(
        {
            "book_first_ppl": "{:.2f}",
            "book_block_ppl": "{:.2f}",
            "run_first_ppl": "{:.2f}",
            "run_block_ppl": "{:.2f}",
            "delta_first_ppl": "{:+.2f}",
            "delta_block_ppl": "{:+.2f}",
            "pct_error_first": "{:+.2f}%",
            "pct_error_block": "{:+.2f}%",
        }
    )
    if hasattr(styler, "map"):
        styled = styler.map(highlight_delta, subset=["delta_first_ppl", "delta_block_ppl"])
    else:
        styled = styler.applymap(highlight_delta, subset=["delta_first_ppl", "delta_block_ppl"])

    styled
except ModuleNotFoundError:
    merged_rows


In [ ]:
model_names = list(comparison["models"])
first_ppls = [comparison["models"][name]["eval"]["first_offset_ppl"] for name in model_names]
block_ppls = [comparison["models"][name]["eval"]["block_ppl"] for name in model_names]
train_histories = [comparison["models"][name]["history"]["train_loss"] for name in model_names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = range(len(model_names))
bar_width = 0.38

axes[0].bar([i - bar_width / 2 for i in x], first_ppls, width=bar_width, label="first-PPL", color="#4c78a8")
axes[0].bar([i + bar_width / 2 for i in x], block_ppls, width=bar_width, label="block-PPL", color="#f58518")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(model_names, rotation=20, ha="right")
axes[0].set_title(f"{comparison['corpus']} structured LM metrics")
axes[0].set_ylabel("Perplexity")
axes[0].set_yscale("log")
axes[0].legend()

for model_name, history in zip(model_names, train_histories):
    axes[1].plot(range(1, len(history) + 1), history, marker="o", label=model_name)
axes[1].set_title("Training loss")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Cross-entropy")
axes[1].legend()

plt.tight_layout()
plt.show()
